# Bridge — clustering × CVR / efficiency

Chạy pipeline RAM-safe: `analysis_cluster_cvr_bridge.py` (DuckDB + chunk session).

**Cần:** outputs clustering đã có; `dim_listing/`, `fact_user_events/`.

**Export:** `outputs/eda_category_1010_1020/bridge/`

In [1]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
cmd = [sys.executable, str(ROOT / "analysis_cluster_cvr_bridge.py")]
# Bỏ comment nếu OOM:
# cmd += ["--event-sample", "0.15", "--memory", "1500MB", "--session-chunk", "30000"]
subprocess.run(cmd, check=True, cwd=ROOT)

Layer 0: register clustering CSVs …
  scope_items=133,718  scope_users=80,000
Layer 1: pos_items (full events, scoped item_id) …
  pos_items=133,717
Layer 1b: pos_users (scoped user_id) …
  pos_users=79,918
Layer 2: listing CVR bridge …
  wrote 01_cvr_by_health_segment.csv (6 rows)
  wrote 02_cvr_by_listing_cluster.csv (10 rows)
  wrote 02b_cvr_health_x_cluster.csv (23 rows)
  wrote 02c_cvr_by_adtype_health.csv (12 rows)
Layer 3: user positive rate by cluster …
  wrote user cluster tables (10 rows)
Layer 3b: event efficiency by health / cluster …
Layer 4: marketplace health ranked …
  wrote 10_health_ranked_underexposed_1010.csv (500 rows)
  wrote 10_health_ranked_underexposed_1020.csv (500 rows)
Layer 5: session funnel (chunksize=50,000) …
  … processed 250,000 session rows
  … processed 500,000 session rows
  wrote 05_session_funnel_by_cluster.csv (122 groups, 672,451 sessions)
Layer 6: insights summary …
  wrote 00_bridge_insights.md
Done. Outputs → /Users/dothinh_3112/Downloads/Dat

CompletedProcess(args=['/Users/dothinh_3112/Downloads/Datathon_Data/env/bin/python', '/Users/dothinh_3112/Downloads/Datathon_Data/analysis_cluster_cvr_bridge.py'], returncode=0)

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

BRIDGE = Path("outputs/eda_category_1010_1020/bridge")
display(Markdown((BRIDGE / "00_bridge_insights.md").read_text()))
for name in (
    "04_segment_event_efficiency.csv",
    "05_session_funnel_by_cluster.csv",
):
    p = BRIDGE / name
    display(Markdown(f"**{name}**"))
    display(pd.read_csv(p))

# Bridge insights — clustering × performance
**Lưu ý cohort:** `20_marketplace_health` chỉ gồm tin có event trong sample clustering → CVR listing (≥1 positive) trên cohort này ~100%. So sánh segment bằng **event efficiency** (contact/pageview, exposure), và session funnel.
**CVR catalog (dim toàn bộ):** xem `outputs/eda_category_1010_1020/02_cvr_baseline_adtype.csv`.

## Event efficiency theo health_segment
- **1010** `high_quality_underexposed`: n=2,328, avg contact/pageview=304.17%, med exposure=2.0
- **1010** `normal`: n=33,014, avg contact/pageview=163.37%, med exposure=4.0
- **1010** `oversaturated_low_conversion`: n=2,092, avg contact/pageview=61.43%, med exposure=8.0
- **1020** `high_quality_underexposed`: n=6,187, avg contact/pageview=270.95%, med exposure=2.0
- **1020** `normal`: n=85,802, avg contact/pageview=145.27%, med exposure=4.0
- **1020** `oversaturated_low_conversion`: n=4,295, avg contact/pageview=58.15%, med exposure=13.0

## CVR listing trên cohort clustering (tham khảo)
- **1010** `high_quality_underexposed`: n=2,328, CVR=100.0%
- **1010** `normal`: n=33,014, CVR=100.0%
- **1010** `oversaturated_low_conversion`: n=2,092, CVR=100.0%
- **1020** `high_quality_underexposed`: n=6,187, CVR=100.0%
- **1020** `normal`: n=85,802, CVR=100.0%
- **1020** `oversaturated_low_conversion`: n=4,295, CVR=100.0%

## Baseline dim (toàn catalog)
- **1010** let: CVR=18.83% (n=454,716)
- **1010** sell: CVR=23.0% (n=157,107)
- **1020** let: CVR=20.45% (n=426,019)
- **1020** sell: CVR=19.32% (n=1,081,845)

## Session funnel (cluster_id=-1 vs search-heavy)
- **1010** noise: contact=91.8%, search=4.9%, deep_compare=51.2%
- **1020** noise: contact=90.8%, search=4.0%, deep_compare=60.2%
- cat=1010 cl=0: search=100%, contact=0%, deep_compare=89%
- cat=1010 cl=1: search=100%, contact=100%, deep_compare=76%
- cat=1020 cl=0: search=100%, contact=0%, deep_compare=91%


**04_segment_event_efficiency.csv**

,category,health_segment,n,avg_event_contact_rate_pct,med_event_contact_rate_pct,avg_exposure,med_exposure,avg_pageviews,avg_unique_users,avg_repeat_viewer_pct,avg_median_age_days
0,1010,high_quality_underexposed,2328,304.17,280.00,1.94,2.0,7.96,22.34,1.14,89.2
1,1010,normal,33014,163.37,146.15,5.20,4.0,12.68,25.92,2.60,39.2
2,1010,oversaturated_low_conversion,2092,61.43,66.43,10.92,8.0,16.35,22.44,5.06,12.6
3,1020,high_quality_underexposed,6187,270.95,247.06,1.92,2.0,7.91,20.79,1.43,57.2
4,1020,normal,85802,145.27,131.25,7.03,4.0,13.71,26.36,2.88,27.1
5,1020,oversaturated_low_conversion,4295,58.15,61.54,20.10,13.0,23.09,31.14,4.91,10.7


**05_session_funnel_by_cluster.csv**

,category,cluster_id,n_sessions,rate_has_contact,rate_has_search,rate_deep_compare,rate_bounce_v2
0,1010,-1,154944,0.9182,0.0493,0.5124,0.0
1,1010,0,456,0.0000,1.0000,0.8882,0.0
2,1010,1,1156,1.0000,1.0000,0.7595,0.0
3,1010,2,117,0.0000,0.0000,1.0000,0.0
4,1010,3,479,0.0000,0.0000,0.9770,0.0
...,...,...,...,...,...,...,...
117,1020,53,444,1.0000,0.0000,0.0000,0.0
118,1020,54,725,1.0000,0.0000,0.9186,0.0
119,1020,55,1967,1.0000,0.0000,0.9380,0.0
120,1020,56,352,1.0000,0.0000,0.0000,0.0
